# Learning module 10 — Start Appium and ask for its status

**Small milestone:** start the local Appium HTTP server, send `GET /status`, read its response, then stop the process this cell started. No Simulator session is created.

The request path is simply **Python → Appium HTTP server → status response**. XCUITest does not receive a device command here. In the installed Appium 3.9.0 source, `getStatus()` returns the server's readiness flag, message, and build metadata; it does not start WebDriverAgent or inspect Movies.

### Dependency follow-up before this check

Fresh `npm audit --json` results on **November 2, 2026 (America/Chicago)**:

| Installation | Moderate affected-package findings | High / critical |
| --- | --- | --- |
| `server/` | 9 | 0 / 0 |
| `server/.tools/appium/` (managed driver home) | 18 | 0 / 0 |

Both reports trace to two underlying advisories; their affected dependency chains account for the larger package counts. These counts overlap between installations and are not 27 independent vulnerabilities. `glob` has a cancellation/traversal issue, and `minimatch` has a pattern-amplification denial-of-service issue. The published fixed versions are 14.0.2 and 11.0.1 respectively; our inspected copies are 13.0.6 and 10.2.6. [glob advisory](https://github.com/advisories/GHSA-3g3m-2fjr-6pw4), [minimatch advisory](https://github.com/advisories/GHSA-g3hv-75wg-m249)

**The findings remain unresolved.** The audit's suggested major-version changes do not preserve our selected server/driver pairing, so no automatic downgrade or unverified dependency override was applied. Audit snapshots are saved under the ignored `server/.tools/audits/` directory.

For this bounded exercise, the inspected extension-discovery path uses a fixed local package glob, and `getStatus()` constructs a status object. We submit no app archive, external glob pattern, or session request. This limited source inspection is not a complete reachability/security audit. Binding to loopback reduces network exposure; it does not patch dependencies or add authentication. Revisit the findings before expanding execution or exposure.

### What the cell does

1. Require the local Node/Appium activation from modules 5 and 6.
2. Check that `127.0.0.1:4723` is free. Leave any existing server alone.
3. Start the exact local executable with an explicit configuration and capture its logs.
4. Poll only `GET /status` for up to 30 seconds and verify `ready` and the expected version.
5. Use `finally` to stop and reap only the child process we started, even if the check fails.

Appium's default listening address is `0.0.0.0`, so this exercise explicitly selects `127.0.0.1`. The empty explicit config avoids inheriting an unrelated Appium config file. No CORS, insecure features, or plugins are enabled by this configuration. [Server CLI options](https://appium.io/docs/en/latest/reference/cli/server/)

**Run this notebook's Node/Appium setup cells first.** Session options from module 9 are not sent by this cell. Logs and the empty config are local, ignored artifacts under `.tools/logs/`.


## This notebook

Part **10 — Start Appium, inspect status, and stop** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Own localhost server; no device session.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.

### Setup for this notebook

These source-only prerequisite cells are repeated intentionally so the lesson is runnable in a fresh kernel. They configure local values/environment and inspect tool versions; they do not launch an Appium server, create a session, or tap/type in the app. The original lessons remain linked for explanation.


In [1]:
# Setup: Local Node activation (lesson 05)
import os
from pathlib import Path
import shutil
import subprocess

notebook_cwd = Path.cwd().resolve()
server_candidates = (notebook_cwd, notebook_cwd / "server", *notebook_cwd.parents)
server_dir = next((candidate for candidate in server_candidates
                   if (candidate / ".node-version").is_file()
                   and (candidate / "package.json").is_file()
                   and (candidate / "sample_scenarios").is_dir()), None)
if server_dir is None:
    raise RuntimeError("Run from the project root, server, or sample_scenarios folder.")

node_version = (server_dir / ".node-version").read_text().strip()
node_bin = server_dir / ".tools" / f"node-v{node_version}-darwin-arm64" / "bin"
if not (node_bin / "node").is_file():
    raise FileNotFoundError(f"Project-local Node is missing: {node_bin}")

path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(node_bin)]
os.environ["PATH"] = os.pathsep.join([str(node_bin), *path_entries])

print("Selected Node:", shutil.which("node"))
print("Node version:", subprocess.check_output(["node", "--version"], text=True, timeout=10).strip())
print("npm version:", subprocess.check_output(["npm", "--version"], text=True, timeout=10).strip())


Selected Node: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/node-v24.22.2-darwin-arm64/bin/node
Node version: v24.22.2
npm version: 11.19.1


In [2]:
# Setup: Local Appium activation (lesson 06)
import json
import os
import shutil
import subprocess

if "server_dir" not in globals() or "node_bin" not in globals():
    raise RuntimeError("Run the Node setup cell in this notebook first.")
if shutil.which("node") != str(node_bin / "node"):
    raise RuntimeError("Reactivate the project-local Node runtime first.")

appium_bin_dir = server_dir / "node_modules" / ".bin"
appium_executable = appium_bin_dir / "appium"
if not appium_executable.is_file():
    raise FileNotFoundError("The project-local Appium package is missing.")

os.environ["APPIUM_HOME"] = str(server_dir / ".tools" / "appium")
os.environ["npm_config_cache"] = str(server_dir / ".tools" / "npm-cache")
path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(appium_bin_dir)]
os.environ["PATH"] = os.pathsep.join([str(appium_bin_dir), *path_entries])

expected_version = json.loads((server_dir / "package.json").read_text())["devDependencies"]["appium"]
actual_version = subprocess.check_output(
    [str(appium_executable), "--version"], cwd=server_dir, text=True, timeout=30
).strip()
if actual_version != expected_version:
    raise RuntimeError(f"Expected Appium {expected_version}, found {actual_version}.")

print("Selected Appium:", shutil.which("appium"))
print("Appium version:", actual_version)
print("Appium home:", os.environ["APPIUM_HOME"])


Selected Appium: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/node_modules/.bin/appium
Appium version: 3.9.0
Appium home: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium


dbug Appium Creating hash file directory: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium/node_modules/.cache/appium


In [3]:
import json
import os
import shutil
import socket
import subprocess
import tempfile
import time
import urllib.error
import urllib.request

if not all(name in globals() for name in ("server_dir", "node_bin", "appium_executable")):
    raise RuntimeError("Run the Node/Appium setup cells in this notebook first.")
if shutil.which("node") != str(node_bin / "node"):
    raise RuntimeError("Rerun this notebook's Node setup cell.")
if os.environ.get("APPIUM_HOME") != str(server_dir / ".tools" / "appium"):
    raise RuntimeError("Rerun this notebook's Appium setup cell.")

host, port = "127.0.0.1", 4723
with socket.socket() as probe:
    probe.bind((host, port))  # A busy port raises; we never stop its existing owner.

expected_version = json.loads((server_dir / "package.json").read_text())["devDependencies"]["appium"]
log_dir = server_dir / ".tools" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", prefix="status-config-",
                                 dir=log_dir, delete=False) as config_file:
    json.dump({"server": {}}, config_file)
status_url = f"http://{host}:{port}/status"
status_opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
process = None
with tempfile.NamedTemporaryFile(mode="w", suffix=".log", prefix="status-check-",
                                 dir=log_dir, delete=False) as log_file:
    print("Server log:", log_file.name)
    try:
        process = subprocess.Popen(
            [str(node_bin / "node"), str(appium_executable),
             "--config", config_file.name, "--address", host, "--port", str(port),
             "--log-no-colors", "--log-timestamp"],
            cwd=server_dir, env=os.environ.copy(),
            stdout=log_file, stderr=subprocess.STDOUT,
        )
        print("Owned Appium PID:", process.pid)
        deadline = time.monotonic() + 30
        last_error = None
        while time.monotonic() < deadline:
            if process.poll() is not None:
                raise RuntimeError(f"Appium exited early; inspect {log_file.name}")
            try:
                with status_opener.open(status_url, timeout=2) as response:
                    status_payload = json.load(response)
                break
            except urllib.error.URLError as error:
                last_error = error
                time.sleep(0.25)
        else:
            raise TimeoutError(f"Appium status timed out; inspect {log_file.name}") from last_error
        value = status_payload.get("value", {})
        if value.get("ready") is not True or value.get("build", {}).get("version") != expected_version:
            raise RuntimeError(f"Unexpected status response: {status_payload}")
        if process.poll() is not None:
            raise RuntimeError("The owned Appium process exited during the status check.")
        print("GET", status_url)
        print(json.dumps(status_payload, indent=2))
    finally:
        if process is not None:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait(timeout=5)
            print("Owned Appium process stopped:", process.poll() is not None)


Server log: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/logs/status-check-5jir_mmi.log
Owned Appium PID: 4670
GET http://127.0.0.1:4723/status
{
  "value": {
    "ready": true,
    "message": "The server is ready to accept new connections",
    "build": {
      "version": "3.9.0",
      "git-sha": "cbd22f516d01dafbd8ed0b3503afac66398228b2",
      "built": "2026-11-02 14:26:32 -0600"
    }
  }
}
Owned Appium process stopped: True


### Read the status response, then pause

- `value.ready: true` is the running server's readiness response at that instant.
- `value.build.version` identifies the responding Appium build. Compare it with our locked version.
- The PID and captured startup log identify the process we started; cleanup applies only to that process.
- `Owned Appium process stopped: True` confirms the child has exited. This cell deliberately leaves no Appium server running.

A later readiness probe can therefore report connection refused until we start Appium again. The stored response is evidence from this bounded check, not a continuously live status indicator.

We have advanced from **installed CLI → running HTTP server → verified status → stopped process**. There is still no session ID, accepted device capabilities, screenshot, UI action, or Interstellar pass/fail result. The dependency findings above remain open.

**Pause here:** the next small milestone is to revalidate the Simulator/app target, start Appium for a device session, and request an XCUITest session using our prepared options. That step can build/start WebDriverAgent and launch or reuse Movies; it belongs to a separate exercise.


**Recorded execution:** `2026-11-02T21:08:20-06:00` using the project Python interpreter. Appium returned `ready: true` and build `3.9.0`; the owned process exited with code `0` after cleanup. The full startup/status log is `/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/logs/status-check-uk_ff5o2.log` (ignored local artifact).


[Index](README.md) · [Previous lesson](09_ios_session_options.ipynb) · [Next lesson](11_ios_session.ipynb)